# 3. CropHarvest: small candidate batches and annual TIFFs

Retain the existing source-label, geometry, identity and observation-year checks. Select at most 25 new eligible parcels per available crop before preparing masks and requesting imagery. Upstream reserved splits remain excluded from development sampling. Full reference discovery is separate from this bounded satellite collection.

Each selected tile retains all 24 half-month periods, so four-month windows can change later. After the candidate batches complete, notebook 04 applies the existing timing/quality checks and reports usable pixels and parcels by locked train/validation/test partition. Subsequent batches fill only measured deficits.

Annual TIFFs are organized by source, year, campaign and round. Each contains 168 labelled bands; sampling masks are separate. Opening the notebook starts no work. Restart Kernel → Run All prepares/rebuilds masks and submits or collects the current small batch. Batch output reports saved data and cloud job states; completion percentages do not represent elapsed processing time.


Data: https://zenodo.org/records/10251170 (download: labels.geojson)

Paper: https://datasets-benchmarks-proceedings.neurips.cc/paper/2021/file/54229abfcfa5649e7003b83dd4755294-Paper-round2.pdf

Repository: https://github.com/nasaharvest/cropharvest

**Output:** screened references and audits are saved in a versioned local folder. Satellite collection and sign-in are enabled for the next Run All. The final archive has all 24 half-month composites per reference year and actual-date vegetation summaries.


## Run mode and preservation

**Restart Kernel → Run All** downloads labels only if missing, applies the existing label checks, then collects full-year satellite composites. `download_missing_labels`, `run_satellite_collection` and `authenticate_now` are enabled. For a fully offline run, disable all three; valid labels must already be present.

Source outputs use versioned folders under `annual_archive_root`. Existing labels, raw files, imagery and old results are preserved. There are no pilot pixel caps in this full-year collection. Timing selection happens in notebook 04.


## 1. Setup

The first cell prepares the cloned package and installs missing dependencies in the active kernel before scientific imports. No prior dataset files or local settings are required.


In [ ]:
# Standard-library setup runs before scientific imports, so a fresh clone needs no saved data.
from pathlib import Path
import sys

notebook_repository_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
notebook_source_folder = str(notebook_repository_root / "src")
if notebook_source_folder not in sys.path:
    sys.path.insert(0, notebook_source_folder)
from crop_pipeline import notebook_setup
notebook_setup.prepare_environment(notebook_repository_root)

from pathlib import Path
from contextlib import contextmanager, redirect_stdout
import io
import json
import os
import re
import zipfile
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shapefile
from IPython.display import display
from shapely.geometry import Point, shape
from shapely.strtree import STRtree

from crop_pipeline.imagery import bilinear as bilinear_collection
from crop_pipeline.imagery.quality import save_json
from crop_pipeline.sources import cropharvest as cropharvest_source
from crop_pipeline.sources import francecrops as francecrops_workflow
from crop_pipeline.sources import sampling as pilot_sampling


Local settings are created automatically when absent. Existing paths are preserved. The optional country-boundary ZIP is not required: dataset-scoped identities remain available when it is absent.


In [ ]:
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
repository_root = Path.cwd()
settings_path = repository_root / "settings.local.json"
local_settings = notebook_setup.ensure_local_settings(settings_path)
labels_path = repository_root / "data/external/cropharvest/labels.geojson"
country_boundaries_path = repository_root / "data/external/cropharvest/ne_50m_admin_0_countries.zip"


Working folders use the local `annual_archive_root` setting. Label content, parameters and helper code determine separate run revisions, so old results remain separate.


In [ ]:
from crop_pipeline.sources import annual
from crop_pipeline.sources import adaptive
run_name = adaptive.COLLECTION_NAME
cropharvest_root = (
    Path(local_settings.get("cropharvest_root", "./cropharvest")).expanduser().resolve()
)
report_folder = (
    Path(local_settings.get("report_root", "./sen4agrinet_work/inventory")).expanduser().resolve()
)
earth_engine_project = os.environ.get("EE_PROJECT", local_settings.get("earth_engine_project", ""))
# Collection starts only when these cells execute; opening the notebook does not retrieve data.
download_missing_labels = True
run_satellite_collection = True
# Small request groups reduce overhead without changing the imagery recipe.
collection_backend = "batch"  # Earth Engine assets; rerun to collect finished jobs
archive_format = "pixel_tables"  # Annual selected-pixel Parquet; raster archives remain available
tile_start = 0
tile_count = None  # Adaptive candidates are already bounded by parcel budgets
download_workers = 2
intervals_per_request = 6
authenticate_now = True
# Explicit decisions for January 1 placeholders or contradictory metadata, keyed by source UID.
observation_year_overrides = {}

annual_archive_root = annual.archive_root(local_settings, report_folder)


# Resolve account setup before large reference transfers start.
earth_engine_project = notebook_setup.project_for_collection(local_settings, settings_path, required=run_satellite_collection)

from crop_pipeline import progress as pipeline_progress
notebook_progress = pipeline_progress.start_notebook(
    "03_cropharvest_preparation",
    ['Obtain labels and configure run', 'Load reference labels', 'Build label table', 'Resolve observation years', 'Screen geometry and duplicates', 'Select target crops', 'Read optional boundaries', 'Assess label reliability', 'Apply eligibility rules', 'Save label audit', 'Extract parcel footprints', 'Prepare eligible parcels', 'Collect annual imagery'],
    report_folder.parent / "progress",
    workload={"folder": str(annual_archive_root), "satellite_collection": globals().get("run_satellite_collection"), "training_mode": globals().get("training_mode")},
)


Shared imagery and quality settings:

- Use every eligible resolved reference year in the configured range.
- Cloud-probability threshold: 40; usable-observation coverage: at least 80%.
- Inward buffer: 10 m.
- Retain all usable pixels; no pilot crop cap.


In [ ]:
minimum_reference_year, maximum_reference_year = 2017, 2100
crop_order = ["cabbage", "carrot", "potato"]
minimum_label_count_for_preview = 100  # A diagnostic only; it does not approve timing.
sampling_seed = 42
cloud_probability_threshold = 40
minimum_valid_fraction = 0.8
usable_pixels_per_crop = None  # The full study retains every usable pixel.
inward_buffer_metres = 10



## Download missing labels and verify the input

`download_missing_labels = True` is set for Run All. If `labels.geojson` already exists, validate and reuse it without downloading or overwriting it. Otherwise retrieve **only the labels** from pinned Zenodo release 10251170, verify its published MD5 checksum and GeoJSON structure, and save it atomically. Interrupted or invalid downloads cannot become the input file.

**Output:** `Labels: downloaded` or `Labels: reused`, feature count and SHA-256. Newly downloaded labels get a `.download.json` provenance record. This step runs before label-content hashing and the versioned run is created. No satellite/feature archive is fetched by this step.


In [ ]:
from crop_pipeline import progress as pipeline_progress
pipeline_progress.begin_stage('Obtain labels and configure run')

import ssl
import certifi

# Set trusted certificates before the first label request, rather than waiting for EE login.
os.environ.setdefault("SSL_CERT_FILE", certifi.where())
print("Trusted certificates:", ssl.create_default_context().cert_store_stats()["x509_ca"])
labels_download = cropharvest_source.ensure_labels(labels_path, download_if_missing=download_missing_labels)
print("Labels:", labels_download["status"], "| features:", labels_download["features"])
print("Label SHA-256:", labels_download["sha256"])

from crop_pipeline.windows.seasons import file_sha256
from crop_pipeline.imagery.retrieval import REQUEST_VERSION, RESAMPLING_POLICY

run_configuration = {
    "labels_sha256": file_sha256(labels_path),
    "country_boundaries_sha256": (
        file_sha256(country_boundaries_path) if country_boundaries_path.exists() else None
    ),
    "year_range": [minimum_reference_year, maximum_reference_year],
    "observation_year_overrides": observation_year_overrides,
    "archive_policy": "all_eligible_parcels_full_calendar_year",
    "sampling_seed": sampling_seed,
    "inward_buffer_metres": inward_buffer_metres,
    "cloud_probability_threshold": cloud_probability_threshold,
    "minimum_valid_fraction": minimum_valid_fraction,
    "usable_pixels_per_crop": usable_pixels_per_crop,
    "request_version": REQUEST_VERSION,
    "resampling": RESAMPLING_POLICY,
    "notebook_source_sha256": pipeline_progress.processing_notebook_fingerprint(
        repository_root / "notebooks/03_cropharvest_preparation.ipynb"
    ),
    "source_helper_sha256": file_sha256(Path(cropharvest_source.__file__)),
    "batch_helper_sha256": file_sha256(
        repository_root / "src/crop_pipeline/imagery/batch_export.py"
    ),
}
run_folder, run_revision = cropharvest_source.initialize_run(
    annual_archive_root / "cropharvest_label_review", run_name, run_configuration
)
source_handoff_folder = run_folder / "source_handoffs"
imagery_root = run_folder / "imagery"
screened_labels_output = run_folder / "labels/target_crop_labels_screened.csv"
window_features_output = run_folder / "analysis/selected_window_features.csv"


def year_folders(year):
    """Keep every output of this immutable collection under its own observation year."""
    return {
        "field": run_folder / "fields" / str(year),
        "l2a_reports": run_folder / "l2a" / str(year),
        "imagery": imagery_root / str(year),
        "timelines": run_folder / "analysis" / str(year) / "timelines",
    }


pd.set_option("display.max_colwidth", 80)
print("Labels:", labels_path)
print("Versioned run:", run_folder)
print("Satellite collection enabled:", run_satellite_collection)


pipeline_progress.end_stage("completed")


Helper: the pipeline prints one line per field group (hundreds of lines like `5493e4… observed summaries reused`). `quiet(...)` sends those lines to a log file and prints a single summary line instead.

In [ ]:
@contextmanager
def quiet(step_name):
    """Hide per-field progress lines; save them to logs/<step_name>.log."""
    log_path = run_folder / "logs" / f"{step_name}.log"
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with open(log_path, "w", buffering=1) as log:  # line-buffered: you can follow it live
        with redirect_stdout(log):
            yield
    lines = log_path.read_text().splitlines()
    problems = [line for line in lines if "failed" in line or "error" in line.lower()]
    note = f"{len(problems)} problem lines, e.g. {problems[0]!r}" if problems else "no failures"
    print(f"[{step_name}] {len(lines):,} progress lines saved to logs/{log_path.name} | {note}")


## 2. Load the labels

**Question:** what is in the CropHarvest label file?

Read the GeoJSON file.

In [ ]:
from crop_pipeline import progress as pipeline_progress
pipeline_progress.begin_stage('Load reference labels')

with labels_path.open(encoding="utf-8") as label_stream:
    source_geojson = json.load(label_stream)
raw_reference_features = source_geojson["features"]

pipeline_progress.end_stage("completed")


Turn it into a table: one row per label, with its shape, a unique ID, dates and a lower-case crop name.

In [ ]:
from crop_pipeline import progress as pipeline_progress
pipeline_progress.begin_stage('Build label table')

label_table = pd.DataFrame(
    [reference_feature["properties"] for reference_feature in raw_reference_features]
)
label_table["geometry"] = [
    shape(reference_feature["geometry"]) if reference_feature.get("geometry") else None
    for reference_feature in raw_reference_features
]
label_table["geom_type"] = [
    reference_geometry.geom_type if reference_geometry is not None else "missing"
    for reference_geometry in label_table["geometry"]
]
label_table["uid"] = label_table["dataset"].astype(str) + ":" + label_table["index"].astype(str)

for column in ["collection_date", "planting_date", "harvest_date", "export_end_date"]:
    label_table[column] = pd.to_datetime(label_table[column], errors="coerce")
label_table["label_norm"] = label_table["label"].astype("string").str.strip().str.casefold()

original_label_table = (
    label_table.copy()
)  # untouched copy, so later cells can start from the original

pipeline_progress.end_stage("completed")


Summary of the raw file.

In [ ]:
display(
    pd.DataFrame(
        {
            "metric": [
                "Labels",
                "Source datasets",
                "Distinct crop names",
                "Labels with a crop name",
                "Polygons",
                "Points",
                "Missing collection date",
            ],
            "value": [
                len(label_table),
                label_table.dataset.nunique(),
                label_table.label_norm.nunique(),
                label_table.label_norm.notna().sum(),
                label_table.geom_type.eq("Polygon").sum(),
                label_table.geom_type.eq("Point").sum(),
                label_table.collection_date.isna().sum(),
            ],
        }
    )
)


## 3. Clean the labels

**Question:** which labels are clearly cabbage, carrot or potato — or clearly another crop, or not a crop?

Every label gets one of five classes: `cabbage`, `carrot`, `potato`, `other_crop`, `non_crop`. Vague names like "vegetables" become `unknown_crop` and are dropped. `chou` counts as cabbage, the same rule as FranceCrops (`CHU`). Small fields are **not** removed here; the 10 m buffer handles that later.

**3a. Resolve observation years, then keep years 2019–2025.**

A January 1 placeholder supplies only a season year when that year agrees with the export-period start; it is not treated as a planting date. Conflicting or missing metadata stays pending unless resolved through `observation_year_overrides`, keyed by source UID. The export-end date is coverage metadata, not a planting date. Every decision is saved in `observation_year_audit.csv`.

In [ ]:
from crop_pipeline import progress as pipeline_progress
pipeline_progress.begin_stage('Resolve observation years')

label_table = original_label_table.copy()
original_count = len(label_table)
for column in ["collection_date", "export_end_date"]:
    label_table[column] = pd.to_datetime(label_table[column], errors="coerce")
label_table["is_crop"] = pd.to_numeric(label_table["is_crop"], errors="coerce")
label_table["label_norm"] = (
    label_table["label"].astype("string").str.strip().str.casefold().replace("", pd.NA)
)
year_decisions = [
    cropharvest_source.resolve_observation_year(
        row.collection_date, row.export_end_date, observation_year_overrides.get(row.uid)
    )
    for row in label_table.itertuples()
]
label_table["observation_year"] = pd.array(
    [decision["year"] for decision in year_decisions], dtype="Int64"
)
label_table["year_resolution_status"] = [decision["status"] for decision in year_decisions]
label_table["year_resolution_reason"] = [decision["reason"] for decision in year_decisions]
year_audit_path = run_folder / "labels/observation_year_audit.csv"
year_audit_path.parent.mkdir(parents=True, exist_ok=True)
label_table[
    [
        "uid",
        "collection_date",
        "export_end_date",
        "observation_year",
        "year_resolution_status",
        "year_resolution_reason",
    ]
].to_csv(year_audit_path, index=False)
valid_year = (
    label_table["observation_year"]
    .between(minimum_reference_year, maximum_reference_year)
    .fillna(False)
)
label_table = label_table.loc[valid_year].copy()
print("Year-resolved labels in range:", len(label_table), "of", original_count)
print("Year decisions:", year_audit_path)

pipeline_progress.end_stage("completed")


**3b. Name variants for our three crops** (English and French).

In [ ]:
target_crop_aliases = {
    "potato": {
        "potato": "",
        "potatoes": "",
        "pomme de terre de consommation": "table potato (FR)",
        "pomme de terre féculière": "starch potato (FR)",
    },
    "carrot": {
        "carrot": "",
        "carrots": "",
        "carotte": "carrot (FR)",
    },
    "cabbage": {
        "cabbage": "",
        "chou": "cabbage (FR); FranceCrops CHU",
    },
}


**3c. Give each label its crop name** (empty if it is not one of the three).

In [ ]:
review_label_aliases = {}

alias_to_crop = {
    alias: crop for crop, variants in target_crop_aliases.items() for alias in variants
}
alias_notes = {
    alias: note for variants in target_crop_aliases.values() for alias, note in variants.items()
}

label_table["crop"] = label_table["label_norm"].map(alias_to_crop).astype("string")
label_table["crop_note"] = label_table["label_norm"].map(alias_notes).fillna("")


**3d. Flag mixed or intercropped labels.** Two crops in one field give a mixed satellite signal.

In [ ]:
mixed_label_pattern = (
    r"\bmix(?:ed|ture|ing)?\b|"
    r"\binter[\s-]?crop\w*\b|"
    r"\bm[ée]lange\w*\b|"
    r"\bassociation\b|"
    r"\bpolyculture\b|"
    r"\bassoc\w*\b|"
    r"\bwith other crops\b"
)
multi_crop_labels = {
    "café / cacao",
    "cauliflower and brocoli",
    "canola/rapeseed (with other crops)",
}

label_table["mixed_label"] = (
    label_table["label_norm"].str.contains(mixed_label_pattern, regex=True, na=False)
    | label_table["label_norm"].isin(multi_crop_labels)
).fillna(False)
print("Mixed labels:", int(label_table["mixed_label"].sum()))


**3e. List vague crop names.** These could hide a cabbage, carrot or potato field, so they cannot be used as `other_crop`.

In [ ]:
generic_crop_labels = {
    "annual crop",
    "vegetables",
    "vegetables (undiff)",
    "leafy or stem vegetable",
    "root",
    "root crops",
    "root, bulb or tuberous vegetable",
    "root/tuber crop with high starch or inulin content",
    "other",
    "other agriculture (undiff)",
    "market gardening",
    "autre légume ou fruit annuel",
    "légume sous abri",
} | set(review_label_aliases)


**3f. Give every label one of the five classes.**

In [ ]:
# Start everything as unknown.
label_table["model_class"] = pd.Series("unknown_crop", index=label_table.index, dtype="string")

# Not a crop (kept even without a name).
label_table.loc[label_table["is_crop"].eq(0), "model_class"] = "non_crop"

# One of our three crops.
target_mask = label_table["is_crop"].eq(1) & label_table["crop"].notna()
label_table.loc[target_mask, "model_class"] = label_table.loc[target_mask, "crop"]

# A clearly named crop that is not one of ours.
other_mask = (
    label_table["is_crop"].eq(1)
    & label_table["label_norm"].notna()
    & label_table["crop"].isna()
    & ~label_table["label_norm"].isin(generic_crop_labels)
)
label_table.loc[other_mask, "model_class"] = "other_crop"

display(label_table["model_class"].value_counts().rename("labels").to_frame())


**3g. Find conflicts:** the exact same field shape, same season, but two different crop names.

In [ ]:
from crop_pipeline import progress as pipeline_progress
pipeline_progress.begin_stage('Screen geometry and duplicates')

label_table["geom_wkb"] = [
    (
        reference_geometry.wkb_hex
        if reference_geometry is not None and not reference_geometry.is_empty
        else None
    )
    for reference_geometry in label_table.geometry
]

identity = label_table["label_norm"].copy()
identity.loc[label_table["is_crop"].eq(0)] = "non_crop"

check = pd.DataFrame(
    {
        "geom": label_table["geom_wkb"],
        "season": label_table["export_end_date"],
        "identity": identity,
    },
    index=label_table.index,
)
valid = (
    check["geom"].notna()
    & check["season"].notna()
    & check["identity"].notna()
    & ~label_table["mixed_label"]
)

names_per_field = check.loc[valid].groupby(["geom", "season"])["identity"].nunique()
conflict_keys = set(names_per_field[names_per_field > 1].index)
label_table["season_conflict"] = [
    (geo, season) in conflict_keys
    for geo, season in zip(label_table["geom_wkb"], label_table["export_end_date"])
]
print("Labels in a conflict:", int(label_table["season_conflict"].sum()))

pipeline_progress.end_stage("completed")


**3h. Keep only the clean labels.**

In [ ]:
keep = (
    ~label_table["mixed_label"]
    & ~label_table["season_conflict"]
    & label_table["model_class"].ne("unknown_crop")
    & label_table["export_end_date"].notna()
    & label_table["geom_wkb"].notna()
)
lab_clean = label_table.loc[keep].copy()
print(f"Clean labels: {len(lab_clean):,}")


**3i. How many labels survived each step?**

In [ ]:
stages = [
    ("Original dataset", original_count),
    ("After year filter", len(label_table)),
    ("After mixed-label filter", int((~label_table["mixed_label"]).sum())),
    (
        "After conflict filter",
        int((~label_table["mixed_label"] & ~label_table["season_conflict"]).sum()),
    ),
    (
        "After geometry/date check",
        int(
            (
                ~label_table["mixed_label"]
                & ~label_table["season_conflict"]
                & label_table["geom_wkb"].notna()
                & label_table["export_end_date"].notna()
            ).sum()
        ),
    ),
    ("Final five-class dataset", len(lab_clean)),
]
filter_summary = pd.DataFrame(stages, columns=["Stage", "Records"])
filter_summary["Removed"] = (-filter_summary["Records"].diff()).fillna(0).astype(int)
filter_summary["Kept (%)"] = (filter_summary["Records"] / original_count * 100).round(2)
display(filter_summary)


Classes before and after cleaning.

In [ ]:
class_comparison = (
    pd.concat(
        [
            label_table["model_class"].value_counts().rename("Before"),
            lab_clean["model_class"].value_counts().rename("After"),
        ],
        axis=1,
    )
    .fillna(0)
    .astype(int)
)
class_comparison["Removed"] = class_comparison["Before"] - class_comparison["After"]
display(class_comparison)


## 4. Keep only field outlines (polygons)

**Question:** how many of our crop labels are polygons? A point has no area, so it cannot be shrunk by 10 m and is dropped.

Take the three target crops and note each label's shape type.

In [ ]:
from crop_pipeline import progress as pipeline_progress
pipeline_progress.begin_stage('Select target crops')

target_labels = lab_clean.loc[lab_clean["crop"].isin(crop_order)].copy()
target_labels["geom_type"] = target_labels["geometry"].apply(
    lambda reviewed_labels: getattr(reviewed_labels, "geom_type", None)
)

display(
    target_labels.groupby("crop")
    .agg(
        labels=("crop", "size"),
        sources=("dataset", "nunique"),
        polygons=("geom_type", lambda series: int(series.isin(["Polygon", "MultiPolygon"]).sum())),
        points=("geom_type", lambda series: int(series.isin(["Point", "MultiPoint"]).sum())),
        reserved_test=("is_test", lambda series: int(series.eq(True).sum())),
    )
    .reindex(crop_order, fill_value=0)
)

pipeline_progress.end_stage("completed")


Which sources the dropped points come from.

In [ ]:
is_polygon = target_labels["geom_type"].isin(["Polygon", "MultiPolygon"])
display(
    pd.crosstab(
        target_labels.loc[~is_polygon, "crop"], target_labels.loc[~is_polygon, "dataset"]
    ).reindex(crop_order, fill_value=0)
)


Drop the points.

In [ ]:
target_labels = target_labels.loc[is_polygon].copy()
print(f"Polygon labels kept: {len(target_labels):,}")


## 5. Where are the fields?

**Question:** which countries do the labels come from? If each crop comes mostly from one country, a model might learn the country instead of the crop.

Load country borders and make a lookup: (lon, lat) → country name.

In [ ]:
from crop_pipeline import progress as pipeline_progress
pipeline_progress.begin_stage('Read optional boundaries')

if country_boundaries_path.exists():
    with zipfile.ZipFile(country_boundaries_path) as archive:
        shape_bytes = io.BytesIO(archive.read("ne_50m_admin_0_countries.shp"))
        attribute_bytes = io.BytesIO(archive.read("ne_50m_admin_0_countries.dbf"))
        reader = shapefile.Reader(shp=shape_bytes, dbf=attribute_bytes)
        country_shapes = [
            (shape(record.shape.__geo_interface__), record.record["NAME"])
            for record in reader.iterShapeRecords()
        ]
    polygons = [geometry for geometry, _ in country_shapes]
    tree = STRtree(polygons)
    geometry_indices = {id(geometry): index for index, geometry in enumerate(polygons)}


def polygon_index(candidate):
    """Support either spatial-index return format without guessing the country."""
    if isinstance(candidate, (int, np.integer)):
        return int(candidate)
    matched_index = geometry_indices.get(id(candidate))
    if matched_index is not None:
        return matched_index
    return next(index for index, geometry in enumerate(polygons) if geometry.equals(candidate))


def country_of(longitude, latitude):
    """Use the saved boundaries, retaining the original nearest-country fallback."""
    point = Point(float(longitude), float(latitude))
    for candidate in tree.query(point):
        country_index = polygon_index(candidate)
        if polygons[country_index].contains(point):
            return country_shapes[country_index][1]
    return country_shapes[polygon_index(tree.nearest(point))][1]

pipeline_progress.end_stage("completed")


Give every label a country (Réunion and Martinique are named by their dataset) and a 1° grid cell.

In [ ]:
if country_boundaries_path.exists():
    TERRITORY = {"reunion-france": "Réunion (FR)", "martinique-france": "Martinique (FR)"}
    target_labels["country"] = [
        TERRITORY.get(dataset) or country_of(lon, lat)
        for dataset, lon, lat in zip(target_labels.dataset, target_labels.lon, target_labels.lat)
    ]
else:
    target_labels["country"] = target_labels["dataset"].astype("string")
    print("Country borders ZIP not found: dataset names used instead.")

target_labels["cell_1deg"] = (
    np.floor(target_labels.lat).astype(int).astype(str)
    + ","
    + np.floor(target_labels.lon).astype(int).astype(str)
)


Labels per country and crop.

In [ ]:
country_crop = pd.crosstab(target_labels.country, target_labels.crop).reindex(
    columns=crop_order, fill_value=0
)
display(country_crop.assign(total=country_crop.sum(axis=1)).sort_values("total", ascending=False))

axis = country_crop.loc[country_crop.sum(axis=1).sort_values(ascending=False).index].plot(
    kind="bar", stacked=True, figsize=(11, 4)
)
axis.set(title="Where the CropHarvest target labels come from", xlabel="Country", ylabel="Labels")
axis.legend(title="Crop")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()


How often do two of our crops grow in the same 1° area?

In [ ]:
shared = (target_labels.groupby("cell_1deg").crop.nunique() >= 2).sum()
print(f"1° cells with at least two target crops: {shared}")


## 6. Field size and label source

**Question:** how trustworthy is each source, and how big are the fields?

How each source collected its labels, and a trust grade (a judgement, not a check of the crop).

In [ ]:
source_methods = {
    "jecam": ("in-situ field survey (JECAM sites)", "high"),
    "canada": ("AAFC ground-truth points (Annual Crop Inventory)", "high"),
    "ile-de-france": ("farmer parcel declarations (French RPG)", "high"),
    "reunion-france": ("farmer parcel declarations (French RPG)", "high"),
    "martinique-france": ("farmer parcel declarations (French RPG)", "high"),
    "kenya": ("field survey", "high"),
    "croplands": ("GFSAD / croplands reference samples (field + image interpretation)", "medium"),
}


Field area (approximate, from degrees), broken shapes, and duplicates.

In [ ]:
from crop_pipeline import progress as pipeline_progress
pipeline_progress.begin_stage('Assess label reliability')

reviewed_labels = target_labels.copy()
meters_per_degree = 111_320.0
reviewed_labels["area_ha"] = [
    (
        reference_geometry.area * meters_per_degree**2 * np.cos(np.radians(lat)) / 1e4
        if kind == "Polygon"
        else np.nan
    )
    for reference_geometry, lat, kind in zip(
        reviewed_labels.geometry, reviewed_labels.lat, reviewed_labels.geom_type
    )
]
reviewed_labels["invalid_geom"] = [
    kind == "Polygon" and not reference_geometry.is_valid
    for reference_geometry, kind in zip(reviewed_labels.geometry, reviewed_labels.geom_type)
]

# Same shape twice in one source / in several sources.
reviewed_labels["dup_in_source"] = reviewed_labels.duplicated(["dataset", "geom_wkb"], keep=False)
dup_source_count = label_table.groupby("geom_wkb").dataset.nunique()
reviewed_labels["dup_across_sources"] = reviewed_labels.geom_wkb.map(dup_source_count).gt(1)

# Crop group in the file differs from what we expect.
expected_group = {
    "potato": "root_tuber",
    "carrot": "vegetables_melons",
    "cabbage": "vegetables_melons",
}
reviewed_labels["group_mismatch"] = reviewed_labels.classification_label.ne(
    reviewed_labels.crop.map(expected_group)
)

# Same field labelled differently in another season.
other_season_count = (
    label_table.dropna(subset=["label_norm"]).groupby("geom_wkb").label_norm.nunique()
)
reviewed_labels["relabelled_other_season"] = reviewed_labels.geom_wkb.map(other_season_count).gt(1)

pipeline_progress.end_stage("completed")


Season dating: does the label have a planting date, an in-season observation, only a year, or dates that do not fit?

In [ ]:
reviewed_labels["placeholder_date"] = reviewed_labels.collection_date.dt.month.eq(
    1
) & reviewed_labels.collection_date.dt.day.eq(1)
start = reviewed_labels.export_end_date - pd.DateOffset(months=12)
inside = reviewed_labels.collection_date.between(start, reviewed_labels.export_end_date)
same_year = reviewed_labels.collection_date.dt.year.eq(start.dt.year)

reviewed_labels["season_class"] = np.select(
    [
        reviewed_labels.planting_date.notna(),
        inside & ~reviewed_labels.placeholder_date,
        reviewed_labels.placeholder_date & (inside | same_year),
    ],
    ["planting / harvest dated", "in-season observation", "season-year only"],
    default="inconsistent",
)
reviewed_labels["season_year"] = start.dt.year


Flag sources whose planting dates are nearly all the same day (likely filler dates), and attach the source grade.

In [ ]:
planting_days_by_source = reviewed_labels.groupby("dataset").planting_date.transform(
    lambda series: series.dropna().dt.strftime("%m-%d").nunique()
)
reviewed_labels["planting_date_placeholder_suspected"] = (
    reviewed_labels.planting_date.notna() & planting_days_by_source.le(2)
)

reviewed_labels["source_method"] = reviewed_labels.dataset.map(
    lambda x: source_methods.get(x, ("unknown", "unknown"))[0]
)
reviewed_labels["source_grade"] = reviewed_labels.dataset.map(
    lambda x: source_methods.get(x, ("unknown", "unknown"))[1]
)


One row per crop and source.

In [ ]:
display(
    reviewed_labels.groupby(["crop", "dataset"]).agg(
        labels=("uid", "size"),
        source_grade=("source_grade", "first"),
        median_area_ha=("area_ha", "median"),
        year_only=("season_class", lambda series: int(series.eq("season-year only").sum())),
        planting_date_present=("planting_date", lambda series: int(series.notna().sum())),
        suspected_placeholder=("planting_date_placeholder_suspected", "sum"),
    )
)


## 7. Final label check and save

**Question:** does each label pass the source checks? A label passes if its source is rated high, its shape is valid, the same shape is not in another source, and its dates make sense.

Mark each label as passed, or give the first reason it failed.

In [ ]:
from crop_pipeline import progress as pipeline_progress
pipeline_progress.begin_stage('Apply eligibility rules')

reviewed_labels["reliable"] = (
    reviewed_labels.source_grade.eq("high")
    & ~reviewed_labels.invalid_geom
    & ~reviewed_labels.dup_across_sources
    & reviewed_labels.season_class.ne("inconsistent")
)

checks = [
    ("Source not high grade", ~reviewed_labels.source_grade.eq("high")),
    ("Invalid polygon", reviewed_labels.invalid_geom),
    ("Duplicate across sources", reviewed_labels.dup_across_sources),
    ("Season inconsistent", reviewed_labels.season_class.eq("inconsistent")),
]
reviewed_labels["fail_reason"] = "passed"
for reason, mask in reversed(checks):  # reversed, so the FIRST failing check wins
    reviewed_labels.loc[mask.fillna(False), "fail_reason"] = reason
assert reviewed_labels.reliable.equals(
    reviewed_labels.fail_reason.eq("passed")
), "Check screening and exclusion reasons"

pipeline_progress.end_stage("completed")


Passed vs. excluded per crop.

In [ ]:
verdict = reviewed_labels.groupby("crop").agg(
    candidates=("uid", "size"), passed=("reliable", "sum"), sources=("dataset", "nunique")
)
verdict["excluded"] = verdict.candidates - verdict.passed
verdict["passed_pct"] = (verdict.passed / verdict.candidates * 100).round(1)
verdict["enough_for_pilot"] = verdict.passed >= minimum_label_count_for_preview
verdict = verdict.reindex(crop_order)
display(verdict)

funnel = pd.crosstab(reviewed_labels.crop, reviewed_labels.fail_reason).reindex(
    index=verdict.index, fill_value=0
)
reason_order = ["passed"] + [name for name, _ in checks if name in funnel]
funnel = funnel.reindex(columns=reason_order, fill_value=0)
axis = funnel.plot.barh(stacked=True, figsize=(11, 4), width=0.72)
axis.set(xlabel="Labels", ylabel="Crop", title="Label check: passed vs. excluded")
axis.invert_yaxis()
axis.legend(title="Result", fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


Save all labels (passed and excluded) with the `reliable` column as the verdict.

In [ ]:
from crop_pipeline import progress as pipeline_progress
pipeline_progress.begin_stage('Save label audit')

original_columns = [
    column
    for column in reviewed_labels.columns
    if column
    not in {
        "geometry",
        "geom_wkb",
        "source_method",
        "source_grade",
        "planting_date_placeholder_suspected",
        "fail_reason",
    }
]
to_export = reviewed_labels[original_columns]
screened_labels_output.parent.mkdir(parents=True, exist_ok=True)
to_export.to_csv(screened_labels_output, index=False)
print(f"Saved {len(to_export):,} labels to {screened_labels_output.name}")

pipeline_progress.end_stage("completed")


---
# Part 2 — Preparing the pixels

## 8. Shrink fields by 10 m and count the pixels inside

**Question:** after shrinking each field 10 m from its edge, how many clean 10 m pixels are left — enough for 500 per crop? This is checked **before** any Earth Engine call.

## 9. Prepare all eligible reference footprints

Retain the existing reliability checks and exclude upstream test labels from development. Polygon footprints with a resolved observation year are eligible. A source date identifies the reference year; it is not treated as a planting date.

**Output:** `reference_features` contains every eligible declared carrot, cabbage or potato parcel. No 100-pixel quota or candidate budget is used.


In [ ]:
from crop_pipeline import progress as pipeline_progress
pipeline_progress.begin_stage('Extract parcel footprints')

labels_revision = "labels.geojson sha256:" + file_sha256(labels_path)
eligible_labels = reviewed_labels.loc[
    reviewed_labels["reliable"].eq(True) & ~reviewed_labels["is_test"].eq(True)
].copy()
eligible_labels["year"] = eligible_labels["observation_year"].astype(int)
reference_features, footprint_exclusions = [], []
for record in eligible_labels.itertuples():
    try:
        reference_features.append(
            cropharvest_source.parcel_feature(
                record.uid,
                record.geometry,
                record.crop,
                record.year,
                record.country,
                record.dataset,
                labels_revision,
            )
        )
    except ValueError as error:
        footprint_exclusions.append({"uid": record.uid, "reason": str(error)})

pipeline_progress.end_stage("completed")


Shrink by 10 m, list the pixels left inside each field, and drop fields with none.

## 10. Plan the full-year archive

Apply the existing 10 m inward buffer and remove contested locations. Keep every eligible interior pixel. Merge same-parcel fragments and tile large footprints without splitting their training identity.

**Output:** `estimate` reports parcels, tiles, image files, uncompressed GiB and available space. All resolved reference years are included. Source-label audits remain in the separate review run.


In [ ]:
from crop_pipeline import progress as pipeline_progress
pipeline_progress.begin_stage('Prepare eligible parcels')

annual_plan, adaptive_context = adaptive.prepare_batch(
    "cropharvest", reference_features, annual_archive_root, report_folder,
    provenance={"labels_revision": labels_revision, "label_review_revision": run_revision,
                "label_review_folder": str(run_folder), "footprint_exclusions": footprint_exclusions},
    inward_buffer_metres=inward_buffer_metres,
)
print("Estimate:", annual_plan["estimate"] if annual_plan else "No new candidates required")
print("Years:", list(annual_plan["years"]) if annual_plan else [])

pipeline_progress.set_workload({"folder": str(annual_archive_root), "source_revision": annual_plan["revision"] if annual_plan else "targets_met_or_exhausted", "satellite_collection": run_satellite_collection})

pipeline_progress.end_stage("completed")


## Collect the full calendar year — enabled for Run All

`run_satellite_collection = True` and `authenticate_now = True` are set for the next Run All. To prepare only, set both to `False`. Earth Engine applies the existing cloud masks, bilinear B5/B11 resampling, daily mosaics and half-month medians. It also calculates dated NDVI, NDRE and NDMI summaries. GeoTIFFs and summary tables are saved locally, with grouped requests and verified-file reuse on restart.

**Output: `status: disabled`** means no satellite authentication, image queries, tasks or imagery downloads ran; label retrieval has its own switch. **`completed`** means every planned tile has all 24 saved intervals and daily-summary checkpoints. A cloudy interval can be saved successfully and still fail the window notebook's quality checks.


In [ ]:
import ssl
import certifi

# Use the system's explicit trust setting when one exists; otherwise use certifi.
os.environ.setdefault("SSL_CERT_FILE", certifi.where())
print("Trusted certificates:", ssl.create_default_context().cert_store_stats()["x509_ca"])


### Collect a bounded candidate batch

Each source notebook selects at most 25 new eligible parcels per crop, spread across geographic/year strata, before rasterization. Annual histories remain available for changing the four-month window. Re-run the same source notebook to collect its existing jobs. After the candidate batches finish, run notebook 04 to count complete usable windows and write deficits. The next run selects new parcels only for deficient crop/partition combinations. Targets are 1,200 training pixels from at least 60 parcels, 400 validation pixels from at least 20 parcels, and 400 reserved-test pixels from at least 20 parcels per crop, with at most 20 retained pixels per parcel. These are initial sampling targets, not a guarantee of model accuracy.

Completed output is organized under `sen4agrinet_work/annual_tifs/<source>/<year>/<campaign>/round_<number>/`. One TIFF per tile contains 168 labelled bands (24 periods × seven measurements). Surrounding unselected pixels are nodata. The compact Parquet files remain the export transport/cache; window analysis reads the organized annual TIFFs. Sampling masks are separate and regenerated during source preparation if removed.

No 5,000-tile or full-catalogue download is the default. Source batches and earlier completed rounds remain separate and are combined by notebook 04. Upstream reserved splits are excluded from development sampling, and locked parcel assignments do not change when new batches are added.


In [ ]:
from crop_pipeline import progress as pipeline_progress
pipeline_progress.begin_stage('Collect annual imagery')

collection_result = adaptive.collect_batch(
    annual_plan, adaptive_context, earth_engine_project,
    enabled=run_satellite_collection, authenticate=authenticate_now,
    tile_start=tile_start, tile_count=tile_count,
    download_workers=download_workers, intervals_per_request=intervals_per_request,
)
if collection_result.get("progress_by_year"):
    print({key: collection_result[key] for key in ("status", "selected_batches", "registry") if key in collection_result})
else:
    print(collection_result)

pipeline_progress.end_stage("skipped" if not run_satellite_collection else "completed" if collection_result["status"] == "completed" else "pending")
pipeline_progress.finish_notebook()


## Assess usable samples in notebook 04

Finish any launched candidate batches, then run [04_four_month_windows.ipynb](04_four_month_windows.ipynb). It combines completed rounds, applies the unchanged timing and quality checks, and retains at most 20 usable pixels per parcel. If crop/partition targets are short, run source notebooks again to select new candidates for those deficits. Continue to 05–06 only after targets are met.

Changing timing rules can reuse annual histories. A cycle crossing a reference-year boundary remains excluded unless adjacent-year imagery is explicitly available. Source labels and reserved evaluation assignments are not replaced to fill a quota.
